# Stronger judge: Qwen2.5-7B-Instruct-AWQ

Runs `data/judge_colab.py` with a 7B judge (AWQ 4-bit, fits on a free T4) on both reps files and downloads
`<CONFIG>_judge7b.csv`.

**Before running:** Runtime → Change runtime type → **T4 GPU** (or better). Then Runtime → Run all.
When asked, upload these 3 files from the repo:
- `data/judge_colab.py`
- `data/processed/GSM8K_Llama-3-8B-Instruct_reps.jsonl`
- `data/processed/MATH_Llama-3-8B-Instruct_reps.jsonl`

**Afterwards:** put both downloaded CSVs in `data/processed/` and run `bash run_all.sh` (it picks them up automatically)
or the commands listed under *TODO: 7B judge* in `results/final/EXPERIMENTS.md`.

Same prompt, 8 verdicts per answer, temperature 0.7, exactly as the 3B judge, so the only change is the model.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q vllm pandas

In [ ]:
from google.colab import files
import os
needed = ["judge_colab.py", "GSM8K_Llama-3-8B-Instruct_reps.jsonl", "MATH_Llama-3-8B-Instruct_reps.jsonl"]
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print("Please upload:", missing)
    files.upload()
assert all(os.path.exists(f) for f in needed), [f for f in needed if not os.path.exists(f)]

In [ ]:
%%bash
set -e
for CFG in GSM8K_Llama-3-8B-Instruct MATH_Llama-3-8B-Instruct; do
  echo "=== $CFG ==="
  python judge_colab.py --model Qwen/Qwen2.5-7B-Instruct-AWQ \
      --reps ${CFG}_reps.jsonl --out ${CFG}_judge7b.csv
done

In [ ]:
import pandas as pd
for cfg in ["GSM8K_Llama-3-8B-Instruct", "MATH_Llama-3-8B-Instruct"]:
    d = pd.read_csv(f"{cfg}_judge7b.csv")
    d = d[d.n_total > 0]
    r = d.n_pass / d.n_total
    print(f"{cfg}: rows={len(d)}  false-pass={r[~d.is_correct].mean():.3f}  false-fail={1 - r[d.is_correct].mean():.3f}")

In [ ]:
files.download("GSM8K_Llama-3-8B-Instruct_judge7b.csv")
files.download("MATH_Llama-3-8B-Instruct_judge7b.csv")